In [9]:
import sys
import logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from functools import reduce

from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFECV
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import classification_report

import ee
ee.Initialize(project='mapbiomas-india')

# Add parent directory to path to import the module
sys.path.append('..')
from feature_selection.feature_selection_module import EEFeatureSelectionModule

# Embed standardized logging for transparent workflow tracking
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Plotting aesthetics
sns.set_theme(style="whitegrid")
plt.rcParams['font.size'] = 11

In [10]:
# Load dataset metadata
classwise_samples_scaled_2k_10k = pd.read_csv('data/classwise_samples_scaled_2k_10k.csv')
runs_analysis_df = pd.read_csv('data/groupwise_runs_default_and_scaled_samples.csv')
region_numbers_df = pd.read_csv('data/regions_area.csv')
legends_df = pd.read_csv('data/class_legend.csv')

# Identify the top-performing region from previous analysis
test_region = runs_analysis_df.sort_values(by=['f1_weighted'], ascending=False).iloc[0]
region_code = test_region['region']
region_id = int(region_numbers_df[region_numbers_df['region_code'] == region_code]['region_id'].iloc[0])
sample_version_in = int(test_region['sample_version'])
n_trees = int(test_region['n_trees'])

# Prepare class balancing sample dictionary
balance_samples = list(map(list, dict(zip(
    classwise_samples_scaled_2k_10k['l2_code'],
    classwise_samples_scaled_2k_10k[region_code]
)).items()))

print(f"Selected Region: {region_code} (ID: {region_id})")
print(f"Sample Version: v{sample_version_in} | Random Forest Trees: {n_trees}")

Selected Region: DES (ID: 3)
Sample Version: v7 | Random Forest Trees: 50


In [19]:
CORR_THRESHOLD = 0.85

In [84]:
# Regional and Temporal Parameters
YEARS = [1995, 2005, 2015, 2024]
# YEARS = [2015]

def drop_collinear_features(df, threshold=0.85):
    """Identifies and drops highly correlated features using Spearman rank correlation."""
    corr_matrix = df.corr(method='spearman').abs()
    # Select upper triangle of correlation matrix
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    # Find features with correlation greater than threshold
    to_drop = [column for column in upper.columns if any(upper[column] > threshold)]
    return to_drop

In [13]:
yr = 2015

In [14]:
# 1. Initialize Module & Balance Data
selector = EEFeatureSelectionModule(
    region_id=region_id, 
    year=yr, 
    sample_version_in=sample_version_in, 
    number_of_trees=n_trees
)
_ = selector.load_and_balance_data(balance_samples=balance_samples)


Calculating Level 1 class distributions (this may take a moment)...


In [15]:
drop_props = ee.List(['random', 'system:index', 'new_id', 'year', 'class', 'area_m2', 'area_proportion'])
all_bands = ee.List(selector.training_partition.first().propertyNames()).removeAll(drop_props).getInfo()


In [16]:
try:
    url = selector.training_partition.getDownloadURL(filetype='csv', selectors=all_bands + ['class'])
    df = pd.read_csv(url)
except Exception:
    sampled = selector.training_partition.select(all_bands + ['class']).limit(5000).getInfo()
    df = pd.DataFrame([feat['properties'] for feat in sampled['features']])
    

In [17]:
url

'https://earthengine.googleapis.com/v1/projects/mapbiomas-india/tables/f52ec8e84c5c3d3bf4fa34bdf4e39a06-aa38fdaea87436db508adefb05d93aa1:getFeatures'

In [20]:
X = df.drop(columns=['class', 'system:index'], errors='ignore')
y = df['class']

# 1. Handle Multicollinearity
collinear_to_drop = drop_collinear_features(X, threshold=CORR_THRESHOLD)

In [ ]:
collinear_to_drop

In [23]:
corr_matrix = X.corr(method='spearman').abs()
# Select upper triangle of correlation matrix


In [24]:
corr_matrix

,soil_median_dry,gvs_median_dry,cai_stdDev,shade_median_dry,savi_median_wet,pri_median_dry,ndvi_stdDev,swir2_min,npv_median_wet,npv_stdDev,...,green_stdDev,green_median_texture,swir1_min,npv_median_dry,wefi_median,cloud_median_dry,ndfi_median_wet,gcvi_median_wet,green_median,swir2_median_dry
soil_median_dry,1.000000,0.320014,0.448450,0.875091,0.163289,0.057052,0.364780,0.744218,0.308801,0.258777,...,0.150422,0.312026,0.782551,0.573840,0.413714,0.071746,0.212997,0.149978,0.319615,0.942499
gvs_median_dry,0.320014,1.000000,0.108559,0.033718,0.549407,0.055985,0.210397,0.306109,0.092400,0.015604,...,0.299930,0.082630,0.241209,0.372527,0.652269,0.554942,0.600519,0.602426,0.679523,0.399353
cai_stdDev,0.448450,0.108559,1.000000,0.401001,0.250699,0.057457,0.888230,0.849031,0.288507,0.514023,...,0.498562,0.313553,0.782584,0.341756,0.531985,0.298919,0.546383,0.217638,0.336067,0.579373
shade_median_dry,0.875091,0.033718,0.401001,1.000000,0.319983,0.151720,0.282966,0.613828,0.134381,0.135158,...,0.198228,0.318759,0.703473,0.293317,0.094633,0.102245,0.062233,0.322898,0.148724,0.793675
savi_median_wet,0.163289,0.549407,0.250699,0.319983,1.000000,0.186450,0.458788,0.178138,0.100361,0.103989,...,0.069004,0.024957,0.081884,0.104858,0.593084,0.682564,0.755984,0.983909,0.650194,0.006075
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
cloud_median_dry,0.071746,0.554942,0.298919,0.102245,0.682564,0.040466,0.462528,0.325400,0.180932,0.043875,...,0.297555,0.169029,0.241844,0.245471,0.563048,1.000000,0.556802,0.727730,0.819477,0.322238
ndfi_median_wet,0.212997,0.600519,0.546383,0.062233,0.755984,0.043242,0.680028,0.551115,0.028811,0.217345,...,0.108239,0.117695,0.472280,0.192717,0.725389,0.556802,1.000000,0.758007,0.606385,0.352372
gcvi_median_wet,0.149978,0.602426,0.217638,0.322898,0.983909,0.168316,0.428009,0.174935,0.102400,0.069677,...,0.144717,0.003631,0.078890,0.126030,0.599974,0.727730,0.758007,1.000000,0.695991,0.017785
green_median,0.319615,0.679523,0.336067,0.148724,0.650194,0.172599,0.432020,0.480327,0.035465,0.040048,...,0.266203,0.307155,0.426882,0.284334,0.521640,0.819477,0.606385,0.695991,1.000000,0.488421


In [25]:
corr_matrix.shape

(119, 119)

In [26]:
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

In [27]:
upper

,soil_median_dry,gvs_median_dry,cai_stdDev,shade_median_dry,savi_median_wet,pri_median_dry,ndvi_stdDev,swir2_min,npv_median_wet,npv_stdDev,...,green_stdDev,green_median_texture,swir1_min,npv_median_dry,wefi_median,cloud_median_dry,ndfi_median_wet,gcvi_median_wet,green_median,swir2_median_dry
soil_median_dry,NaN,0.320014,0.448450,0.875091,0.163289,0.057052,0.364780,0.744218,0.308801,0.258777,...,0.150422,0.312026,0.782551,0.573840,0.413714,0.071746,0.212997,0.149978,0.319615,0.942499
gvs_median_dry,NaN,NaN,0.108559,0.033718,0.549407,0.055985,0.210397,0.306109,0.092400,0.015604,...,0.299930,0.082630,0.241209,0.372527,0.652269,0.554942,0.600519,0.602426,0.679523,0.399353
cai_stdDev,NaN,NaN,NaN,0.401001,0.250699,0.057457,0.888230,0.849031,0.288507,0.514023,...,0.498562,0.313553,0.782584,0.341756,0.531985,0.298919,0.546383,0.217638,0.336067,0.579373
shade_median_dry,NaN,NaN,NaN,NaN,0.319983,0.151720,0.282966,0.613828,0.134381,0.135158,...,0.198228,0.318759,0.703473,0.293317,0.094633,0.102245,0.062233,0.322898,0.148724,0.793675
savi_median_wet,NaN,NaN,NaN,NaN,NaN,0.186450,0.458788,0.178138,0.100361,0.103989,...,0.069004,0.024957,0.081884,0.104858,0.593084,0.682564,0.755984,0.983909,0.650194,0.006075
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
cloud_median_dry,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.556802,0.727730,0.819477,0.322238
ndfi_median_wet,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.758007,0.606385,0.352372
gcvi_median_wet,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.695991,0.017785
green_median,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.488421


In [33]:
cols_dropped = [column for column in upper.columns if any(upper[column] > CORR_THRESHOLD)]
cols_dropped

['shade_median_dry',
 'ndvi_stdDev',
 'shade_median',
 'soil_median',
 'gv_amp',
 'hallcover_median',
 'evi2_stdDev',
 'gvs_median',
 'ndvi_median',
 'ndfi_stdDev',
 'savi_median',
 'gcvi_stdDev',
 'green_min',
 'npv_max',
 'gcvi_median_dry',
 'evi2_median_wet',
 'blue_median_dry',
 'swir2_median_wet',
 'red_median_dry',
 'wefi_median_wet',
 'cloud_median_wet',
 'hallcover_stdDev',
 'npv_amp',
 'evi2_amp',
 'gvs_stdDev',
 'soil_stdDev',
 'ndwi_stdDev',
 'savi_median_dry',
 'green_median_wet',
 'blue_min',
 'blue_median',
 'blue_median_wet',
 'shade_amp',
 'gv_min',
 'ndvi_amp',
 'sefi_median',
 'gvs_max',
 'ndvi_median_dry',
 'evi2_median_dry',
 'gv_max',
 'green_median_dry',
 'shade_max',
 'red_stdDev',
 'gvs_min',
 'sefi_median_dry',
 'swir1_median_wet',
 'blue_stdDev',
 'shade_min',
 'swir1_median',
 'swir2_median',
 'red_median',
 'nir_min',
 'gv_median_dry',
 'ndfi_max',
 'ndvi_median_wet',
 'ndwi_amp',
 'cloud_stdDev',
 'ndwi_median_wet',
 'ndfi_median_dry',
 'ndfi_min',
 'evi2_m

Approach 2: Also compute correlation with target and remove those that arent correlating with the target as much 

In [31]:
def improved_correlation_drop(df, target_col, threshold=0.85):
    """
    Drops highly correlated features by keeping the one 
    with a higher correlation to the target variable.
    """
    # 1. Compute correlation matrix for features
    features = df.drop(columns=[target_col])
    corr_matrix = features.corr().abs()
    
    # 2. Compute correlation of all features with the target
    target_corr = df.corr()[target_col].abs()
    
    # Find pairs above threshold (using upper triangle to avoid duplicates)
    upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    
    to_drop = set()
    
    for col in upper_tri.columns:
        # Find features highly correlated with the current column
        correlated_features = upper_tri.index[upper_tri[col] > threshold].tolist()
        
        for feat in correlated_features:
            # Compare target correlations
            if target_corr[col] > target_corr[feat]:
                to_drop.add(feat)
            else:
                to_drop.add(col)
                
    # Drop identified columns
    df_dropped = df.drop(columns=list(to_drop))
    print(f"Dropped {len(to_drop)} features: {list(to_drop)}")
    return df_dropped,to_drop

In [32]:
df_filtered2,cols_dropped2 = improved_correlation_drop(df, target_col='class', threshold=CORR_THRESHOLD)

Dropped 85 features: ['soil_median', 'shade_median', 'nir_min', 'cloud_stdDev', 'ndfi_median_dry', 'green_stdDev', 'wefi_median', 'nir_median_dry', 'gvs_min', 'soil_stdDev', 'npv_stdDev', 'savi_median_dry', 'red_median', 'savi_median_wet', 'gvs_median', 'gvs_stdDev', 'gvs_max', 'swir2_min', 'cloud_max', 'savi_median', 'red_median_dry', 'cloud_amp', 'swir1_median_dry', 'swir1_min', 'shade_stdDev', 'swir1_median_wet', 'npv_median_dry', 'blue_median_dry', 'ndvi_stdDev', 'gv_max', 'gv_stdDev', 'shade_median_wet', 'ndvi_amp', 'swir2_median_wet', 'red_min', 'swir2_median', 'green_min', 'evi2_median_dry', 'gvs_amp', 'wefi_stdDev', 'evi2_median', 'savi_stdDev', 'green_median_wet', 'blue_median_wet', 'gcvi_median', 'gcvi_stdDev', 'green_median', 'cloud_median_dry', 'npv_amp', 'hallcover_median', 'shade_max', 'gv_median_wet', 'ndfi_median_wet', 'gcvi_median_dry', 'swir1_stdDev', 'evi2_amp', 'wefi_median_wet', 'soil_median_dry', 'red_stdDev', 'soil_median_wet', 'gv_median_dry', 'ndfi_max', 'evi2_

Compare the 2 lists

In [35]:
print(len(cols_dropped)),print(len(cols_dropped2))

86
85


(None, None)

In [38]:
len(list(set(cols_dropped) & set(cols_dropped2)))

68

In [39]:
list(set(cols_dropped) - set(cols_dropped2))

['ndvi_median',
 'ndwi_amp',
 'gv_median',
 'shade_min',
 'ndfi_stdDev',
 'soil_max',
 'shade_median_dry',
 'cloud_median_wet',
 'npv_max',
 'cloud_min',
 'green_median_dry',
 'sefi_median_dry',
 'ndfi_min',
 'hallcover_stdDev',
 'shade_amp',
 'ndwi_median_wet',
 'ndfi_median',
 'ndvi_median_dry']

In [22]:
len(collinear_to_drop)

86

In [ ]:
df, target_col, threshold=0.85

In [41]:
X_filtered,_ = improved_correlation_drop(df=df, target_col='class', threshold=CORR_THRESHOLD)

Dropped 85 features: ['soil_median', 'shade_median', 'nir_min', 'cloud_stdDev', 'ndfi_median_dry', 'green_stdDev', 'wefi_median', 'nir_median_dry', 'gvs_min', 'soil_stdDev', 'npv_stdDev', 'savi_median_dry', 'red_median', 'savi_median_wet', 'gvs_median', 'gvs_stdDev', 'gvs_max', 'swir2_min', 'cloud_max', 'savi_median', 'red_median_dry', 'cloud_amp', 'swir1_median_dry', 'swir1_min', 'shade_stdDev', 'swir1_median_wet', 'npv_median_dry', 'blue_median_dry', 'ndvi_stdDev', 'gv_max', 'gv_stdDev', 'shade_median_wet', 'ndvi_amp', 'swir2_median_wet', 'red_min', 'swir2_median', 'green_min', 'evi2_median_dry', 'gvs_amp', 'wefi_stdDev', 'evi2_median', 'savi_stdDev', 'green_median_wet', 'blue_median_wet', 'gcvi_median', 'gcvi_stdDev', 'green_median', 'cloud_median_dry', 'npv_amp', 'hallcover_median', 'shade_max', 'gv_median_wet', 'ndfi_median_wet', 'gcvi_median_dry', 'swir1_stdDev', 'evi2_amp', 'wefi_median_wet', 'soil_median_dry', 'red_stdDev', 'soil_median_wet', 'gv_median_dry', 'ndfi_max', 'evi2_

In [43]:
X.shape

(6000, 119)

In [42]:
X_filtered.shape

(6000, 35)

In [44]:
X_filtered.columns

Index(['cai_stdDev', 'shade_median_dry', 'pri_median_dry', 'npv_median_wet',
       'cai_median', 'nir_median', 'ndvi_median', 'ndfi_stdDev', 'slope',
       'npv_max', 'npv_min', 'soil_amp', 'sefi_stdDev', 'cloud_median_wet',
       'hallcover_stdDev', 'cai_median_dry', 'shade_amp', 'ndwi_median',
       'ndvi_median_dry', 'ndwi_median_dry', 'green_median_dry', 'pri_median',
       'sefi_median_dry', 'shade_min', 'nir_median_wet', 'ndwi_amp',
       'ndwi_median_wet', 'ndfi_min', 'pri_median_wet', 'ndfi_median',
       'cloud_min', 'gv_median', 'soil_max', 'green_median_texture', 'class'],
      dtype='str')

In [46]:
X_filtered.columns.tolist()

['cai_stdDev',
 'shade_median_dry',
 'pri_median_dry',
 'npv_median_wet',
 'cai_median',
 'nir_median',
 'ndvi_median',
 'ndfi_stdDev',
 'slope',
 'npv_max',
 'npv_min',
 'soil_amp',
 'sefi_stdDev',
 'cloud_median_wet',
 'hallcover_stdDev',
 'cai_median_dry',
 'shade_amp',
 'ndwi_median',
 'ndvi_median_dry',
 'ndwi_median_dry',
 'green_median_dry',
 'pri_median',
 'sefi_median_dry',
 'shade_min',
 'nir_median_wet',
 'ndwi_amp',
 'ndwi_median_wet',
 'ndfi_min',
 'pri_median_wet',
 'ndfi_median',
 'cloud_min',
 'gv_median',
 'soil_max',
 'green_median_texture',
 'class']

In [48]:
trained_rf_corr, acc_corr, f1_corr, y_true_corr, y_pred_corr = selector.evaluate_model(X_filtered.columns.tolist())

In [49]:
print(f1_corr)

0.9403953166453629


In [50]:
print(acc_corr)

0.9403578528827038


Comparison with features obtained from other approach

In [51]:
top_30_features = ['pri_median',
 'pri_median_dry',
 'green_median_texture',
 'pri_median_wet',
 'slope',
 'ndvi_median_dry',
 'blue_median_wet',
 'red_median_dry',
 'blue_median',
 'red_median',
 'green_min',
 'cai_median_dry',
 'nir_stdDev',
 'cai_median',
 'blue_median_dry',
 'swir1_min',
 'red_stdDev',
 'green_median',
 'blue_min',
 'savi_median_dry',
 'gcvi_median_dry',
 'ndwi_median_dry',
 'swir2_min',
 'green_median_wet',
 'ndvi_median',
 'cai_stdDev',
 'gcvi_median',
 'savi_median',
 'blue_stdDev',
 'nir_median_dry']

In [52]:
trained_rf_top30, acc_top30, f1_top30, y_true_top30, y_pred_top30 = selector.evaluate_model(top_30_features)

In [53]:
print(f1_top30)

0.9338354669530811


In [54]:
print(acc_top30)

0.9337309476474487


In [ ]:
rf = RandomForestClassifier(n_estimators=n_trees, random_state=42,min_samples_leaf = 5, n_jobs=-1)

In [79]:
filtered_features_yearwise_df = pd.DataFrame(
    {
    'year':[],
    'band':[]
    }
)

# filtered_features_yearwise_df

In [ ]:
yearwise_df = pd.DataFrame({'year': [yr] * len(X_filtered.columns), 'band': X_filtered.columns.tolist()})
yearwise_df

,year,band
0,2015,cai_stdDev
1,2015,shade_median_dry
2,2015,pri_median_dry
3,2015,npv_median_wet
4,2015,cai_median
5,2015,nir_median
6,2015,ndvi_median
7,2015,ndfi_stdDev
8,2015,slope
9,2015,npv_max


In [ ]:
# stable_features_per_year = {}
# feature_ranks_per_year = {}


for yr in YEARS:
    logging.info(f"====== PROCESSING TEMPORAL SNAPSHOT: {yr} ======")
    
    # 1. Initialize Module & Balance Data
    selector = EEFeatureSelectionModule(
        region_id=region_id, 
        year=yr, 
        sample_version_in=sample_version_in, 
        number_of_trees=n_trees
    )
    _ = selector.load_and_balance_data(balance_samples=balance_samples)
    
    # 2. Localize Training Data for Scikit-Learn
    # Exclude non-predictive metadata properties
    drop_props = ee.List(['random', 'system:index', 'new_id', 'year', 'class', 'area_m2', 'area_proportion'])
    all_bands = ee.List(selector.training_partition.first().propertyNames()).removeAll(drop_props).getInfo()
    
    logging.info(f"Downloading training partition for {len(all_bands)} bands...")
    try:
        url = selector.training_partition.getDownloadURL(filetype='csv', selectors=all_bands + ['class'])
        df = pd.read_csv(url)
    except Exception as e:
        logging.warning(f"Direct download failed: {e}. Falling back to 5000-sample limit.")
        sampled = selector.training_partition.select(all_bands + ['class']).limit(5000).getInfo()
        df = pd.DataFrame([feat['properties'] for feat in sampled['features']])
        
    X = df.drop(columns=['class', 'system:index'], errors='ignore')
    y = df['class']
    
    # 3. Filter Method: Remove Multicollinearity
    _, collinear_to_drop = improved_correlation_drop(df, target_col='class', threshold=CORR_THRESHOLD)
    X_filtered = X.drop(columns=collinear_to_drop)
    logging.info(f"Dropped {len(collinear_to_drop)} highly correlated features. {X_filtered.shape[1]} remain.")

    if (X_filtered.shape[1] < 40):
        yearwise_df = pd.DataFrame({'year': [yr] * len(X_filtered.columns), 'band': X_filtered.columns.tolist()})
        filtered_features_yearwise_df = pd.concat([filtered_features_yearwise_df, yearwise_df], ignore_index=True)
    
    # # 4. Embedded Method: RFECV
    # rf = RandomForestClassifier(n_estimators=n_trees, random_state=42,min_samples_leaf = 5, n_jobs=-1)
    # cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    
    # logging.info("Running Recursive Feature Elimination with Cross-Validation...")
    # rfecv = RFECV(estimator=rf, step=1, cv=cv, scoring='f1_weighted', n_jobs=-1)
    # rfecv.fit(X_filtered, y)
    
    # # 5. Extract Results
    # selected_features = X_filtered.columns[rfecv.support_].tolist()
    # stable_features_per_year[yr] = selected_features
    
    # ranks = pd.DataFrame({'band': X_filtered.columns, f'rank_{yr}': rfecv.ranking_})
    # feature_ranks_per_year[yr] = ranks
    
    # logging.info(f"Optimal features identified for {yr}: {rfecv.n_features_}\n")

2026-08-20 18:34:19,095 - INFO - ====== PROCESSING TEMPORAL SNAPSHOT: 1995 ======


Calculating Level 1 class distributions (this may take a moment)...


2026-08-20 18:35:16,735 - INFO - Downloading training partition for 119 bands...
2026-08-20 18:36:05,888 - INFO - Dropped 84 highly correlated features. 35 remain.
2026-08-20 18:36:05,894 - INFO - ====== PROCESSING TEMPORAL SNAPSHOT: 2005 ======


Dropped 84 features: ['soil_median', 'shade_median', 'nir_min', 'cloud_stdDev', 'ndfi_median_dry', 'green_stdDev', 'npv_max', 'ndfi_median', 'gvs_min', 'soil_stdDev', 'npv_stdDev', 'savi_median_dry', 'red_median', 'savi_median_wet', 'gvs_median', 'gvs_stdDev', 'ndfi_stdDev', 'gvs_max', 'swir2_min', 'cloud_max', 'savi_median', 'red_median_dry', 'cloud_amp', 'swir1_median_dry', 'swir1_min', 'shade_stdDev', 'swir1_median_wet', 'blue_median_dry', 'ndvi_stdDev', 'gv_max', 'gv_median', 'gv_stdDev', 'ndwi_median_wet', 'ndwi_median', 'swir2_median_wet', 'red_min', 'swir2_median', 'green_min', 'evi2_median_dry', 'gvs_amp', 'wefi_stdDev', 'evi2_median', 'savi_stdDev', 'green_median_wet', 'blue_median_wet', 'gcvi_median', 'gcvi_stdDev', 'green_median', 'hallcover_stdDev', 'ndvi_median', 'npv_amp', 'hallcover_median', 'gv_median_wet', 'ndfi_median_wet', 'gcvi_median_dry', 'shade_amp', 'evi2_amp', 'wefi_median_wet', 'soil_median_dry', 'red_stdDev', 'soil_median_wet', 'gv_median_dry', 'ndfi_max', 'e

2026-08-20 18:37:15,932 - INFO - Downloading training partition for 119 bands...
2026-08-20 18:38:32,228 - INFO - Dropped 89 highly correlated features. 30 remain.
2026-08-20 18:38:32,235 - INFO - ====== PROCESSING TEMPORAL SNAPSHOT: 2015 ======


Dropped 89 features: ['soil_median', 'shade_median', 'cloud_stdDev', 'ndfi_median_dry', 'green_stdDev', 'wefi_median', 'ndfi_median', 'gvs_min', 'nir_median_wet', 'npv_stdDev', 'savi_median_dry', 'red_median', 'savi_median_wet', 'gvs_median', 'gvs_stdDev', 'ndfi_stdDev', 'gvs_max', 'swir2_min', 'savi_median', 'red_median_dry', 'cloud_amp', 'swir1_median_dry', 'swir1_min', 'shade_stdDev', 'npv_median_dry', 'blue_median_dry', 'ndvi_stdDev', 'gv_max', 'gv_stdDev', 'shade_median_dry', 'ndwi_median_wet', 'ndvi_amp', 'ndwi_median', 'swir2_median_wet', 'red_min', 'swir2_median', 'green_min', 'evi2_median_dry', 'gvs_amp', 'wefi_stdDev', 'evi2_median', 'savi_stdDev', 'green_median_wet', 'blue_median_wet', 'gcvi_stdDev', 'gcvi_median', 'green_median', 'hallcover_stdDev', 'cloud_median_dry', 'cai_median', 'ndvi_median', 'npv_amp', 'hallcover_median', 'gv_median_wet', 'ndfi_median_wet', 'gcvi_median_dry', 'evi2_amp', 'wefi_median_wet', 'soil_median_dry', 'red_stdDev', 'soil_median_wet', 'gv_median

2026-08-20 18:39:07,341 - INFO - Downloading training partition for 119 bands...
2026-08-20 18:40:19,022 - INFO - Dropped 85 highly correlated features. 34 remain.
2026-08-20 18:40:19,028 - INFO - ====== PROCESSING TEMPORAL SNAPSHOT: 2024 ======


Dropped 85 features: ['soil_median', 'shade_median', 'nir_min', 'cloud_stdDev', 'ndfi_median_dry', 'green_stdDev', 'wefi_median', 'nir_median_dry', 'gvs_min', 'soil_stdDev', 'npv_stdDev', 'savi_median_dry', 'red_median', 'savi_median_wet', 'gvs_median', 'gvs_stdDev', 'gvs_max', 'swir2_min', 'cloud_max', 'savi_median', 'red_median_dry', 'cloud_amp', 'swir1_median_dry', 'swir1_min', 'shade_stdDev', 'swir1_median_wet', 'npv_median_dry', 'blue_median_dry', 'ndvi_stdDev', 'gv_max', 'gv_stdDev', 'shade_median_wet', 'ndvi_amp', 'swir2_median_wet', 'red_min', 'swir2_median', 'green_min', 'evi2_median_dry', 'gvs_amp', 'wefi_stdDev', 'evi2_median', 'savi_stdDev', 'green_median_wet', 'blue_median_wet', 'gcvi_median', 'gcvi_stdDev', 'green_median', 'cloud_median_dry', 'npv_amp', 'hallcover_median', 'shade_max', 'gv_median_wet', 'ndfi_median_wet', 'gcvi_median_dry', 'swir1_stdDev', 'evi2_amp', 'wefi_median_wet', 'soil_median_dry', 'red_stdDev', 'soil_median_wet', 'gv_median_dry', 'ndfi_max', 'evi2_

2026-08-20 18:41:18,750 - INFO - Downloading training partition for 119 bands...
2026-08-20 18:42:31,801 - INFO - Dropped 86 highly correlated features. 33 remain.


Dropped 86 features: ['soil_median', 'shade_median', 'nir_min', 'cloud_stdDev', 'green_stdDev', 'npv_max', 'wefi_median', 'nir_median_dry', 'ndfi_median', 'gvs_min', 'soil_stdDev', 'npv_stdDev', 'savi_median_dry', 'red_median', 'savi_median_wet', 'gvs_median', 'gvs_stdDev', 'ndfi_stdDev', 'gvs_max', 'swir2_min', 'cloud_max', 'savi_median', 'red_median_dry', 'cloud_amp', 'swir1_median_dry', 'swir1_min', 'shade_stdDev', 'blue_median_dry', 'ndvi_stdDev', 'gv_max', 'gv_median', 'gv_stdDev', 'shade_median_dry', 'shade_median_wet', 'swir2_median_wet', 'red_min', 'swir2_median', 'green_min', 'evi2_median_dry', 'gvs_amp', 'wefi_stdDev', 'evi2_median', 'savi_stdDev', 'green_median_wet', 'blue_median_wet', 'gcvi_median', 'green_median', 'cloud_median_dry', 'ndvi_median', 'npv_amp', 'hallcover_median', 'shade_max', 'gv_median_wet', 'ndfi_median_wet', 'gcvi_median_dry', 'swir1_stdDev', 'evi2_amp', 'wefi_median_wet', 'soil_median_dry', 'red_stdDev', 'soil_median_wet', 'gv_median_dry', 'ndfi_min', '

In [88]:
filtered_features_yearwise_df.groupby(['year'])['band'].count()

year
1995.0    35
2005.0    30
2015.0    69
2024.0    33
Name: band, dtype: int64

In [91]:
filtered_features_yearwise_df.drop_duplicates(inplace=True)

In [92]:
filtered_features_yearwise_df.groupby(['year'])['band'].count()

year
1995.0    35
2005.0    30
2015.0    35
2024.0    33
Name: band, dtype: int64

In [96]:
features_nyears_df = filtered_features_yearwise_df.groupby(['band'])['year'].count().reset_index()

In [ ]:
features_nyears_df.columns = ['band', 'num_years']

In [ ]:
features_nyears_df.sort_values(by='num_years',ascending=False)

,band,num_years
3,cai_stdDev,4
2,cai_median_dry,4
24,ndwi_amp,4
22,ndvi_median_dry,4
12,green_median_texture,4
38,pri_median,4
37,npv_min,4
36,npv_median_wet,4
28,nir_median,4
26,ndwi_median_dry,4


In [ ]:
features_nyears_df[features_nyears_df['num_years'] >= 3].shape

(22, 2)

In [106]:
features_nyears_df[features_nyears_df['num_years'] >= 3]['band'].to_list()

['cai_median',
 'cai_median_dry',
 'cai_stdDev',
 'cloud_min',
 'green_median_texture',
 'ndfi_min',
 'ndvi_median_dry',
 'ndwi_amp',
 'ndwi_median_dry',
 'nir_median',
 'nir_median_wet',
 'npv_median_wet',
 'npv_min',
 'pri_median',
 'pri_median_dry',
 'pri_median_wet',
 'sefi_median_dry',
 'sefi_stdDev',
 'shade_amp',
 'shade_min',
 'slope',
 'soil_amp']

In [ ]:
# stable_features_per_year = {}
# feature_ranks_per_year = {}

# for yr in YEARS:
#     logging.info(f"====== PROCESSING TEMPORAL SNAPSHOT: {yr} ======")
    
#     # 1. Initialize Module & Balance Data
#     selector = EEFeatureSelectionModule(
#         region_id=region_id, 
#         year=yr, 
#         sample_version_in=sample_version_in, 
#         number_of_trees=n_trees
#     )
#     _ = selector.load_and_balance_data(balance_samples=balance_samples)
    
#     # 2. Localize Training Data for Scikit-Learn
#     # Exclude non-predictive metadata properties
#     drop_props = ee.List(['random', 'system:index', 'new_id', 'year', 'class', 'area_m2', 'area_proportion'])
#     all_bands = ee.List(selector.training_partition.first().propertyNames()).removeAll(drop_props).getInfo()
    
#     logging.info(f"Downloading training partition for {len(all_bands)} bands...")
#     try:
#         url = selector.training_partition.getDownloadURL(filetype='csv', selectors=all_bands + ['class'])
#         df = pd.read_csv(url)
#     except Exception as e:
#         logging.warning(f"Direct download failed: {e}. Falling back to 5000-sample limit.")
#         sampled = selector.training_partition.select(all_bands + ['class']).limit(5000).getInfo()
#         df = pd.DataFrame([feat['properties'] for feat in sampled['features']])
        
#     X = df.drop(columns=['class', 'system:index'], errors='ignore')
#     y = df['class']
    
#     # 3. Filter Method: Remove Multicollinearity
#     collinear_to_drop = drop_collinear_features(X, threshold=0.85)
#     X_filtered = X.drop(columns=collinear_to_drop)
#     logging.info(f"Dropped {len(collinear_to_drop)} highly correlated features. {X_filtered.shape[1]} remain.")
    
#     # 4. Embedded Method: RFECV
#     rf = RandomForestClassifier(n_estimators=n_trees, random_state=42,min_samples_leaf = 5, n_jobs=-1)
#     cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    
#     logging.info("Running Recursive Feature Elimination with Cross-Validation...")
#     rfecv = RFECV(estimator=rf, step=1, cv=cv, scoring='f1_weighted', n_jobs=-1)
#     rfecv.fit(X_filtered, y)
    
#     # 5. Extract Results
#     selected_features = X_filtered.columns[rfecv.support_].tolist()
#     stable_features_per_year[yr] = selected_features
    
#     ranks = pd.DataFrame({'band': X_filtered.columns, f'rank_{yr}': rfecv.ranking_})
#     feature_ranks_per_year[yr] = ranks
    
#     logging.info(f"Optimal features identified for {yr}: {rfecv.n_features_}\n")

KeyboardInterrupt: 